In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

In [ ]:
#imports from langchain

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter

In [ ]:
#imports from langchain for embedding and vectorization

from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.vectorstores import FAISS
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
# imports from langchain for memory and retrieval

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

In [ ]:
# low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [ ]:
# load environment variables
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

# create openai client
openai = OpenAI(api_key=api_key)

# Load Folder directory and documents

In [ ]:
# Read in documents using LangChain's Loaders
# Take everything in all sub-folders of our knowledge base


folders = glob.glob("knowledge-base/*")

documents = []

for folder in folders:
    doc_type = os.path.basename(folder) ## products, contracts, company, employees
    loader = DirectoryLoader(folder, glob = "**/*.md", loader_cls = TextLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)
        


In [ ]:
print(folders)

In [ ]:
len(documents)

In [ ]:
documents[12]

# Documents are Ready !!

# Now break the documents into chunks with overlap

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = text_splitter.split_documents(documents)

In [ ]:
len(chunks)

In [ ]:
chunks[12]

## Check Doc Types from chunks

In [ ]:
doc_types = set(chunk.metadata["doc_type"] for chunk in chunks)
print(doc_types)

In [ ]:
for chunk in chunks:
    if "CEO" in chunk.page_content:
        print(chunk)
        print("\n____________\n")

# Auto Encoding LLMS

## Encoding and Vectorization of data chunks

In [ ]:
# embeddings

embeddings = OpenAIEmbeddings()

# Chroma as Vector DB

In [ ]:
# Check if the Chroma Datastore already exists, if so - delete the collection to start from scratch

if os.path.exists(db_name):
    Chroma(persist_directory = db_name, embedding_function = embeddings).delete_collection()

In [ ]:
# Create our Chroma Vector Store

vectorstore = Chroma.from_documents(documents = chunks, embedding = embeddings, persist_directory = db_name)


In [ ]:
print(f"Vector Store created with {vectorstore._collection.count()} documents"  )

In [ ]:
# Get one vector and find how many dimensions it has

collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include = ["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions} dimensions")

In [ ]:
sample_embedding

## Visualizing the Vector Store

Let's take a minute to look at the documents and their embedding vectors to see what's going on.

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
doc_types = [metadata['doc_type'] for metadata in result['metadatas']]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

# FAISS as alternative to Chroma

In [ ]:
# FAISS Usage Alternative to Chroma
# FAISS -> stores data in ram and not on disk

vectorstore = FAISS.from_documents(chunks, embedding=embeddings)

total_vectors = vectorstore.index.ntotal
dimensions = vectorstore.index.d

print(f"There are {total_vectors} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
# Prework
vectors = []
documents = []
doc_types = []
colors = []
color_map = {'products':'blue', 'employees':'green', 'contracts':'red', 'company':'orange'}

for i in range(total_vectors):
    vectors.append(vectorstore.index.reconstruct(i))
    doc_id = vectorstore.index_to_docstore_id[i]
    document = vectorstore.docstore.search(doc_id)
    documents.append(document.page_content)
    doc_type = document.metadata['doc_type']
    doc_types.append(doc_type)
    colors.append(color_map[doc_type])
    
vectors = np.array(vectors)

## 2D Visualization

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## 3D Visualization

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

# Using Langchain for memory, retrieval and LLM

In [ ]:
# Create langchain abstractions for LLM, memory and chain

# create a new Chat with openAI
llm = ChatOpenAI(temperature = 0.7, model_name = MODEL)

# setup conversation mempry for chat
memory = ConversationBufferMemory(memory_key = 'chat_history', return_messages = True)

# the retriever is abstraction over vectore store which will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: setup the conversation chain with GPT 3.5 LLM, the vectore store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm = llm, retriever = retriever, memory = memory)

In [ ]:
query = "Can you describe Insurellm in few sentences"
result = conversation_chain.invoke({"question" : query})
print(result["answer"])

In [ ]:
query = "Who is Alex"
result = conversation_chain.invoke({"question" : query})
print(result["answer"])

In [ ]:
query = "I am asking about Chen"
result = conversation_chain.invoke({"question" : query})
print(result["answer"])

# Gradio Interface for CHAT

In [ ]:
# Create function for langchain conversation

def chat(message, history): # history we are not using, since langchain maintains its own memory
    result = conversation_chain.invoke({"question" : message})
    return result["answer"]

In [ ]:
# Add Gradio UI

view = gr.ChatInterface(chat).launch()

# Langchain LLM Background Run
## check what goes behind the scenes

### Usage of Callbacks

In [ ]:
from langchain_core.callbacks import StdOutCallbackHandler


# Create langchain abstractions for LLM, memory and chain

# create a new Chat with openAI
llm = ChatOpenAI(temperature = 0.7, model_name = MODEL)

# setup conversation mempry for chat
memory = ConversationBufferMemory(memory_key = 'chat_history', return_messages = True)

# the retriever is abstraction over vectore store which will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: setup the conversation chain with GPT 3.5 LLM, the vectore store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm = llm, retriever = retriever, memory = memory, callbacks = [StdOutCallbackHandler()])

query = "Who won the prestigious IIOTY award in 2023"
result = conversation_chain.invoke({"question" : query})

print(result["answer"])

# The Chunks being passed are 3 and not able to find the proper answer

# We can explicitly mention how many chunks to pass to LLM using search_kwargs = {"k" : 25}

In [ ]:
# the retriever is abstraction over vectore store which will be used during RAG
retriever = vectorstore.as_retriever(search_kwargs = {"k" : 25})

# putting it together: setup the conversation chain with GPT 3.5 LLM, the vectore store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm = llm, retriever = retriever, memory = memory, callbacks = [StdOutCallbackHandler()])

query = "Who won the prestigious IIOTY award in 2023"
result = conversation_chain.invoke({"question" : query})

print(result["answer"])

### This would provide the answer has number of chunks increased and llm got more context